# **1 CONFIG**

## 1.1 Google Sheets

In [ ]:
sheet_link = 'https://docs.google.com/spreadsheets/d/1NCUqs4EoKX1Ker-YR_IBzMP11VkM46jWZoxe2cZ0z0g/edit#gid=791063404'
sheet_name = 'Data LO for testing'
sheet_range = 'A11:R'
result_sheet_name = 'LO Result 1'

## 1.2 Parameters

In [ ]:
# exchange_rate =

# 2 IMPORT PACKAGE

In [ ]:
import math
import pandas as pd
import numpy as np
import datetime

import gspread
import gspread_dataframe as gd

from google.colab import data_table, auth
data_table.enable_dataframe_formatter()
auth.authenticate_user()

from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 3 READ AND CLEAN DATA

## 3.1 Read

In [ ]:
df_raw = pd.DataFrame.from_records(gc.open_by_url(sheet_link).worksheet(sheet_name).get(sheet_range))
df_raw.columns = df_raw.iloc[0,:]
df_raw = df_raw.iloc[1:,:]
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68 entries, 1 to 68
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   SKU              68 non-null     object
 1   Material         68 non-null     object
 2   Description      68 non-null     object
 3   Category         68 non-null     object
 4   Safety           68 non-null     object
 5   Safety value     68 non-null     object
 6   Price (VND)      68 non-null     object
 7   BOM              68 non-null     object
 8   Price ($)        68 non-null     object
 9   MOQ (PM)         68 non-null     object
 10  MOQ (SKU)        68 non-null     object
 11  RV (PM)          68 non-null     object
 12  RV (SKU)         68 non-null     object
 13  Available (PM)   68 non-null     object
 14  Available (SKU)  68 non-null     object
 15  Min Qty (SKU)    68 non-null     object
 16  Max Qty (SKU)    68 non-null     object
 17  Max Scrap ($)    68 non-null     obje

In [ ]:
df_raw.head()

,SKU,Material,Description,Category,Safety,Safety value,Price (VND),BOM,Price ($),MOQ (PM),MOQ (SKU),RV (PM),RV (SKU),Available (PM),Available (SKU),Min Qty (SKU),Max Qty (SKU),Max Scrap ($)
1,CN07168A,P11241795,EC 19 FHS SHIPPER 1PK 72 NA,Shipper,10,2,"4,892",1,$0.21,300,300,20,20,"5,191","5,183",408.5,36584,$90.00
2,731059,P11241929,ZZ 20 FHS SHIPPER 1PK POLYBAG 72 NA,Shipper,10,2,"5,169",1,$0.22,300,300,20,20,"8,465","8,452",1.5,70720,$90.00
3,CN01375A,P11241787,ZZ 20 FHS SHIPPER 1PK 72 NA,Shipper,10,2,"5,257",1,$0.23,300,300,20,20,399,398,0.5,20000,$90.00
4,61029952,P11212226,EC 19 RPP FHM BCARD 1PK T2X6 72 ASIA,Backer card,"2,000",11,132,97,$0.01,"50,400",520,"4,200",43,"19,848",205,660,20000,$90.00
5,61029952,P11280000,EC 19 FHM ZIPPER BAG 1PK 96 ASIA,Plastic Bag,"1,000",29,660,24,$0.03,80000,"3,300",5000,206,"10,441",431,660,20000,$90.00


## 3.2 Clean

In [ ]:
df_raw.columns = df_raw.columns.str.strip()
df_raw = df_raw.astype(str)
df_raw = df_raw.applymap(lambda x: x.strip() if isinstance(x, str) else x)
df_raw = df_raw.replace({'#N/A': None, '#VALUE!': None, '#REF!': None, 'None': None, '': None})

In [ ]:
df_raw.head()

,SKU,Material,Description,Category,Safety,Safety value,Price (VND),BOM,Price ($),MOQ (PM),MOQ (SKU),RV (PM),RV (SKU),Available (PM),Available (SKU),Min Qty (SKU),Max Qty (SKU),Max Scrap ($)
1,CN07168A,P11241795,EC 19 FHS SHIPPER 1PK 72 NA,Shipper,10,2,"4,892",1,$0.21,300,300,20,20,"5,191","5,183",408.5,36584,$90.00
2,731059,P11241929,ZZ 20 FHS SHIPPER 1PK POLYBAG 72 NA,Shipper,10,2,"5,169",1,$0.22,300,300,20,20,"8,465","8,452",1.5,70720,$90.00
3,CN01375A,P11241787,ZZ 20 FHS SHIPPER 1PK 72 NA,Shipper,10,2,"5,257",1,$0.23,300,300,20,20,399,398,0.5,20000,$90.00
4,61029952,P11212226,EC 19 RPP FHM BCARD 1PK T2X6 72 ASIA,Backer card,"2,000",11,132,97,$0.01,"50,400",520,"4,200",43,"19,848",205,660,20000,$90.00
5,61029952,P11280000,EC 19 FHM ZIPPER BAG 1PK 96 ASIA,Plastic Bag,"1,000",29,660,24,$0.03,80000,"3,300",5000,206,"10,441",431,660,20000,$90.00


In [ ]:
numeric_columns = ['Safety', 'Price (VND)', 'BOM', 'Price ($)', 'MOQ (PM)', 'MOQ (SKU)', 'RV (PM)', 'RV (SKU)', 'Available (PM)', 'Available (SKU)','Min Qty (SKU)','Max Qty (SKU)','Max Scrap ($)']

# Replace characters in the specified columns
df_raw[numeric_columns] = df_raw[numeric_columns].replace({',': '', '\$': '', r'\(': '', r'\)': ''}, regex=True)
df_raw[numeric_columns] = df_raw[numeric_columns].apply(pd.to_numeric, errors='coerce')

In [ ]:
df_raw.head()

,SKU,Material,Description,Category,Safety,Safety value,Price (VND),BOM,Price ($),MOQ (PM),MOQ (SKU),RV (PM),RV (SKU),Available (PM),Available (SKU),Min Qty (SKU),Max Qty (SKU),Max Scrap ($)
1,CN07168A,P11241795,EC 19 FHS SHIPPER 1PK 72 NA,Shipper,10,2,4892.0,1,0.21,300.0,300.0,20,20,5191,5183,408.5,36584,90.0
2,731059,P11241929,ZZ 20 FHS SHIPPER 1PK POLYBAG 72 NA,Shipper,10,2,5169.0,1,0.22,300.0,300.0,20,20,8465,8452,1.5,70720,90.0
3,CN01375A,P11241787,ZZ 20 FHS SHIPPER 1PK 72 NA,Shipper,10,2,5257.0,1,0.23,300.0,300.0,20,20,399,398,0.5,20000,90.0
4,61029952,P11212226,EC 19 RPP FHM BCARD 1PK T2X6 72 ASIA,Backer card,2000,11,132.0,97,0.01,50400.0,520.0,4200,43,19848,205,660.0,20000,90.0
5,61029952,P11280000,EC 19 FHM ZIPPER BAG 1PK 96 ASIA,Plastic Bag,1000,29,660.0,24,0.03,80000.0,3300.0,5000,206,10441,431,660.0,20000,90.0


# 4 FUNCTION

In [ ]:
# def find_satisfied_quantity(list_p, qmin, qmax, list_bom, list_price, list_moq, list_rv, list_leftover, max_scrap):
#     for q in range(qmin, qmax+1):
#         scrap_value = 0
#         list_add = []
#         for p in range(len(list_p)):
#             if (list_leftover[p] / list_bom[p]) >= q:
#                 add = 0
#             elif (list_leftover[p] / list_bom[p]) < q:
#                 add = max(math.ceil((q * list_bom[p] - list_leftover[p]) / list_rv[p]) * list_rv[p], list_moq[p])

#             scrap_value += ((list_leftover[p] + add) - q * list_bom[p]) * list_price[p]
#             list_add.append(add)
#         if scrap_value <= max_scrap:
#             return list_add

#     return None

In [ ]:
def find_satisfied_quantity(list_p, qmin, qmax, list_bom, list_price, list_moq, list_rv, list_leftover, max_scrap, safety_qty):
    for q in range(qmin, qmax+1):
        scrap_value = 0
        list_add = []
        for p in range(len(list_p)):
            if ((list_leftover[p]- safety_qty[p]) / list_bom[p]) >= q: #change here
                add = 0
            elif ((list_leftover[p]-safety_qty[p]) / list_bom[p]) < q: #change here
                add = max(math.ceil((q * list_bom[p] + safety_qty[p] - list_leftover[p]) / list_rv[p]) * list_rv[p], list_moq[p]) #change here

            scrap_value += ((list_leftover[p] + add) - q * list_bom[p]) * list_price[p]
            list_add.append(add)
        if scrap_value <= max_scrap:
            return list_add

    return None

In [ ]:
# import math

# result = max(math.ceil((1355 * 25 + 2000 - 11560) / 1200) * 1200, 16800)

# print(result)

In [ ]:
# List of columns to keep
columns_to_keep = ['SKU', 'Material', 'Category', 'Safety', 'Min Qty (SKU)', 'Max Qty (SKU)', 'BOM', 'Price ($)', 'MOQ (PM)', 'RV (PM)', 'Available (PM)','Max Scrap ($)']

# Keep only the specified columns
df = df_raw[columns_to_keep]
df['Min Qty (SKU)'] = df['Min Qty (SKU)'].astype(int)
df['Max Qty (SKU)'] = df['Max Qty (SKU)'].astype(int)
df.head(10)

<ipython-input-54-3b8758733eac>:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Min Qty (SKU)'] = df['Min Qty (SKU)'].astype(int)
<ipython-input-54-3b8758733eac>:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Max Qty (SKU)'] = df['Max Qty (SKU)'].astype(int)


,SKU,Material,Category,Safety,Min Qty (SKU),Max Qty (SKU),BOM,Price ($),MOQ (PM),RV (PM),Available (PM),Max Scrap ($)
1,CN07168A,P11241795,Shipper,10,408,36584,1,0.21,300.0,20,5191,90.0
2,731059,P11241929,Shipper,10,1,70720,1,0.22,300.0,20,8465,90.0
3,CN01375A,P11241787,Shipper,10,0,20000,1,0.23,300.0,20,399,90.0
4,61029952,P11212226,Backer card,2000,660,20000,97,0.01,50400.0,4200,19848,90.0
5,61029952,P11280000,Plastic Bag,1000,660,20000,24,0.03,80000.0,5000,10441,90.0
6,61029952,P11242132,Shipper,10,660,20000,1,0.25,300.0,20,196,90.0
7,61029951,P11211988,Backer card,2000,630,20000,61,0.01,50400.0,3600,5514,90.0
8,61029951,P11242095,Shipper,10,630,20000,1,0.21,300.0,20,8,90.0
9,61029951,P11280001,Plastic Bag,1000,630,20000,12,0.04,65000.0,5000,1823,90.0
10,61029949,P11211974,Backer card,2000,700,20000,36,0.01,50400.0,3600,50400,90.0


In [ ]:
df1 = pd.DataFrame()
for SKU, group in df.groupby('SKU'):
    try:
        list_p = group['Material'].tolist()
        qmin = group['Min Qty (SKU)'].min()
        qmax = group['Max Qty (SKU)'].max()
        list_bom = group['BOM'].tolist()
        list_price = group['Price ($)'].tolist()
        list_moq = group['MOQ (PM)'].tolist()
        list_rv = group['RV (PM)'].tolist()
        list_leftover = group['Available (PM)'].tolist()
        max_scrap = group['Max Scrap ($)'].min()
        safety_qty = group['Safety'].tolist() #change here


        list_add = find_satisfied_quantity(list_p, qmin, qmax, list_bom, list_price, list_moq, list_rv, list_leftover, max_scrap, safety_qty)

        group['Opt 1 Buy (PM)'] = list_add

        df1 = pd.concat([df1, group])

    except Exception as e:
        print(f"Error processing group for SKU {SKU}")
        continue

In [ ]:
df1.head()

,SKU,Material,Category,Safety,Min Qty (SKU),Max Qty (SKU),BOM,Price ($),MOQ (PM),RV (PM),Available (PM),Max Scrap ($),Opt 1 Buy (PM)
23,1051927,P11212008,Backer card,2000,630,20000,25,0.02,16800.0,1200,11560,90.0,24000.0
24,1051927,P11250717,Sticker,2000,630,20000,24,0.01,30000.0,10000,36660,90.0,0.0
25,1051927,P11241748,Shipper,10,630,20000,1,0.27,300.0,20,815,90.0,540.0
52,1051928,P11212280,Backer card,2000,0,20000,25,0.03,16800.0,1200,4660,90.0,25200.0
53,1051928,P11250835,Sticker,2000,0,20000,24,0.01,30000.0,5000,29500,90.0,0.0


In [ ]:
# Tính SKU Qty theo từng P
df1['Opt 1 Qty (SKU)'] = round((df1['Opt 1 Buy (PM)'] + df1['Available (PM)'] - df1['Safety']) / df1['BOM'], 2) #change here

In [ ]:
# SKU Qty sau cùng là min của từng P và round down integer
df1['Opt 1 Actual Qty (SKU)'] = df1.groupby('SKU')['Opt 1 Qty (SKU)'].transform('min')
df1['Opt 1 Actual Qty (SKU)'] = np.floor(df1['Opt 1 Actual Qty (SKU)']).astype(int)

In [ ]:
# Scrap value từng P
df1['Opt 1 Scrap Value'] = (df1['Available (PM)'] + df1['Opt 1 Buy (PM)'] - df1['Opt 1 Actual Qty (SKU)']*df1['BOM'])*df1['Price ($)']

In [ ]:
# Scrap value cho SKU
df1['Opt 1 Total Scrap Value'] = df1.groupby('SKU')['Opt 1 Scrap Value'].transform('sum')

In [ ]:
df1['Opt 1 Scrap Value'] = round(df1['Opt 1 Scrap Value'], 2)
df1['Opt 1 Total Scrap Value'] = round(df1['Opt 1 Total Scrap Value'], 2)

In [ ]:
df1

,SKU,Material,Category,Safety,Min Qty (SKU),Max Qty (SKU),BOM,Price ($),MOQ (PM),RV (PM),Available (PM),Max Scrap ($),Opt 1 Buy (PM),Opt 1 Qty (SKU),Opt 1 Actual Qty (SKU),Opt 1 Scrap Value,Opt 1 Total Scrap Value
23,1051927,P11212008,Backer card,2000,630,20000,25,0.02,16800.0,1200,11560,90.0,24000.0,1342.40,1342,40.20,88.23
24,1051927,P11250717,Sticker,2000,630,20000,24,0.01,30000.0,10000,36660,90.0,0.0,1444.17,1342,44.52,88.23
25,1051927,P11241748,Shipper,10,630,20000,1,0.27,300.0,20,815,90.0,540.0,1345.00,1342,3.51,88.23
52,1051928,P11212280,Backer card,2000,0,20000,25,0.03,16800.0,1200,4660,90.0,25200.0,1114.40,1114,60.30,87.94
53,1051928,P11250835,Sticker,2000,0,20000,24,0.01,30000.0,5000,29500,90.0,0.0,1145.83,1114,27.64,87.94
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42,VN00154A,P11241798,Shipper,10,337,61376,1,0.09,300.0,20,4662,90.0,2380.0,7032.00,7032,0.90,88.66
43,VN00154A,P11250754,Sticker,2000,337,61376,12,0.00,30000.0,5000,92962,90.0,0.0,7580.17,7032,0.00,88.66
37,VN00592A,P11212081,Backer card,2000,0,172480,12,0.01,32000.0,2000,211428,90.0,0.0,17452.33,17172,53.64,54.54
38,VN00592A,P11250748,Sticker,2000,0,172480,12,0.00,30000.0,5000,198410,90.0,30000.0,18867.50,17172,0.00,54.54


In [ ]:
filtered_rows = df1[df1['Opt 1 Total Scrap Value'] > df1['Max Scrap ($)']]
filtered_rows

,SKU,Material,Category,Safety,Min Qty (SKU),Max Qty (SKU),BOM,Price ($),MOQ (PM),RV (PM),Available (PM),Max Scrap ($),Opt 1 Buy (PM),Opt 1 Qty (SKU),Opt 1 Actual Qty (SKU),Opt 1 Scrap Value,Opt 1 Total Scrap Value


In [ ]:
# chạy lần 2

In [ ]:
df2 = pd.DataFrame()
for SKU, group in df1.groupby('SKU'):
    try:
        list_p = group['Material'].tolist()
        qmin = group['Opt 1 Actual Qty (SKU)'].max() + 1 # find next solution
        qmax = group['Max Qty (SKU)'].max()
        list_bom = group['BOM'].tolist()
        list_price = group['Price ($)'].tolist()
        list_moq = group['MOQ (PM)'].tolist()
        list_rv = group['RV (PM)'].tolist()
        list_leftover = group['Available (PM)'].tolist()
        max_scrap = group['Max Scrap ($)'].min()
        safety_qty = group['Safety'].tolist()


        list_add = find_satisfied_quantity(list_p, qmin, qmax, list_bom, list_price, list_moq, list_rv, list_leftover, max_scrap, safety_qty)

        group['Opt 2 Buy (PM)'] = list_add

        df2 = pd.concat([df2, group])

    except Exception as e:
        print(f"Error processing group for SKU {SKU}")
        continue

In [ ]:
# Tính SKU Qty theo từng P
df2['Opt 2 Qty (SKU)'] = round((df2['Opt 2 Buy (PM)'] + df2['Available (PM)'] - df2['Safety'] ) / df2['BOM'], 2) #change here
# SKU Qty sau cùng là min của từng P và round down integer
df2['Opt 2 Actual Qty (SKU)'] = df2.groupby('SKU')['Opt 2 Qty (SKU)'].transform('min')
df2['Opt 2 Actual Qty (SKU)'] = np.floor(df2['Opt 2 Actual Qty (SKU)']).astype(int)
# Scrap value từng P
df2['Opt 2 Scrap Value'] = (df2['Available (PM)'] + df2['Opt 2 Buy (PM)'] - df2['Opt 2 Actual Qty (SKU)']*df2['BOM'])*df2['Price ($)']
# Scrap value cho SKU
df2['Opt 2 Total Scrap Value'] = df2.groupby('SKU')['Opt 2 Scrap Value'].transform('sum')

df2['Opt 2 Scrap Value'] = round(df2['Opt 2 Scrap Value'], 2)
df2['Opt 2 Total Scrap Value'] = round(df2['Opt 2 Total Scrap Value'], 2)

In [ ]:
df2.reset_index(drop=True, inplace=True)
df2

,SKU,Material,Category,Safety,Min Qty (SKU),Max Qty (SKU),BOM,Price ($),MOQ (PM),RV (PM),...,Opt 1 Buy (PM),Opt 1 Qty (SKU),Opt 1 Actual Qty (SKU),Opt 1 Scrap Value,Opt 1 Total Scrap Value,Opt 2 Buy (PM),Opt 2 Qty (SKU),Opt 2 Actual Qty (SKU),Opt 2 Scrap Value,Opt 2 Total Scrap Value
0,1051927,P11212008,Backer card,2000,630,20000,25,0.02,16800.0,1200,...,24000.0,1342.40,1342,40.20,88.23,25200.0,1390.40,1385,42.70,79.60
1,1051927,P11250717,Sticker,2000,630,20000,24,0.01,30000.0,10000,...,0.0,1444.17,1342,44.52,88.23,0.0,1444.17,1385,34.20,79.60
2,1051927,P11241748,Shipper,10,630,20000,1,0.27,300.0,20,...,540.0,1345.00,1342,3.51,88.23,580.0,1385.00,1385,2.70,79.60
3,1051928,P11212280,Backer card,2000,0,20000,25,0.03,16800.0,1200,...,25200.0,1114.40,1114,60.30,87.94,56400.0,2362.40,2362,60.30,88.42
4,1051928,P11250835,Sticker,2000,0,20000,24,0.01,30000.0,5000,...,0.0,1145.83,1114,27.64,87.94,30000.0,2395.83,2362,28.12,88.42
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63,VN00154A,P11241798,Shipper,10,337,61376,1,0.09,300.0,20,...,2380.0,7032.00,7032,0.90,88.66,2400.0,7052.00,7052,0.90,83.86
64,VN00154A,P11250754,Sticker,2000,337,61376,12,0.00,30000.0,5000,...,0.0,7580.17,7032,0.00,88.66,0.0,7580.17,7052,0.00,83.86
65,VN00592A,P11212081,Backer card,2000,0,172480,12,0.01,32000.0,2000,...,0.0,17452.33,17172,53.64,54.54,0.0,17452.33,17272,41.64,42.54
66,VN00592A,P11250748,Sticker,2000,0,172480,12,0.00,30000.0,5000,...,30000.0,18867.50,17172,0.00,54.54,30000.0,18867.50,17272,0.00,42.54


In [ ]:
filtered_rows2 = df2[df2['Opt 2 Total Scrap Value'] > df2['Max Scrap ($)']]
filtered_rows2

,SKU,Material,Category,Safety,Min Qty (SKU),Max Qty (SKU),BOM,Price ($),MOQ (PM),RV (PM),...,Opt 1 Buy (PM),Opt 1 Qty (SKU),Opt 1 Actual Qty (SKU),Opt 1 Scrap Value,Opt 1 Total Scrap Value,Opt 2 Buy (PM),Opt 2 Qty (SKU),Opt 2 Actual Qty (SKU),Opt 2 Scrap Value,Opt 2 Total Scrap Value


In [ ]:
# chạy lần 3

In [ ]:
df3 = pd.DataFrame()
for SKU, group in df2.groupby('SKU'):
    try:
        list_p = group['Material'].tolist()
        qmin = group['Opt 2 Actual Qty (SKU)'].max() + 1 # find next solution
        qmax = group['Max Qty (SKU)'].max()
        list_bom = group['BOM'].tolist()
        list_price = group['Price ($)'].tolist()
        list_moq = group['MOQ (PM)'].tolist()
        list_rv = group['RV (PM)'].tolist()
        list_leftover = group['Available (PM)'].tolist()
        max_scrap = group['Max Scrap ($)'].min()
        safety_qty = group['Safety'].tolist()


        list_add = find_satisfied_quantity(list_p, qmin, qmax, list_bom, list_price, list_moq, list_rv, list_leftover, max_scrap, safety_qty)

        group['Opt 3 Buy (PM)'] = list_add

        df3 = pd.concat([df3, group])

    except Exception as e:
        print(f"Error processing group for SKU {SKU}")
        continue

In [ ]:
# Tính SKU Qty theo từng P
df3['Opt 3 Qty (SKU)'] = round((df3['Opt 3 Buy (PM)'] + df3['Available (PM)'] - df3['Safety'] ) / df3['BOM'], 2) #change here
# SKU Qty sau cùng là min của từng P và round down integer
df3['Opt 3 Actual Qty (SKU)'] = df3.groupby('SKU')['Opt 3 Qty (SKU)'].transform('min')
df3['Opt 3 Actual Qty (SKU)'] = np.floor(df3['Opt 3 Actual Qty (SKU)']).astype(int)
# Scrap value từng P
df3['Opt 3 Scrap Value'] = (df3['Available (PM)'] + df3['Opt 3 Buy (PM)'] - df3['Opt 3 Actual Qty (SKU)']*df3['BOM'])*df3['Price ($)']
# Scrap value cho SKU
df3['Opt 3 Total Scrap Value'] = df3.groupby('SKU')['Opt 3 Scrap Value'].transform('sum')

df3['Opt 3 Scrap Value'] = round(df3['Opt 3 Scrap Value'], 2)
df3['Opt 3 Total Scrap Value'] = round(df3['Opt 3 Total Scrap Value'], 2)

# 5 WRITE RESULT

In [ ]:
result_link = sheet_link
result_wb = gc.open_by_url(result_link)
result_sheet = result_wb.worksheet(result_sheet_name)

# Define the range to clear (from A1 to V{num_rows})
num_rows = result_sheet.row_count
range_to_clear = f'A1:V{num_rows}'

# Clear the specific range
result_sheet.update(range_to_clear, [[]])

# Now, write the dataframe to the cleared range
gd.set_with_dataframe(result_sheet, df3)


In [ ]:
# draft

In [ ]:
def find_satisfied_quantity(list_p, qmin, qmax, list_bom, list_price, list_moq, list_rv, list_leftover):
    for q in range(qmin, qmax+1):
        scrap_value = 0
        list_add = []
        for p in range(len(list_p)):
            if (list_leftover[p] / list_bom[p]) >= q:
                add = 0
            elif (list_leftover[p] / list_bom[p]) < q:
                add = max(math.ceil((q * list_bom[p] - list_leftover[p]) / list_rv[p]) * list_rv[p], list_moq[p])

            scrap_value += ((list_leftover[p] + add) - q * list_bom[p]) * list_price[p]
            list_add.append(add)
        if scrap_value <= 100:
            return list_add

    return None

# Example usage:
list_p = [1, 2, 3, 4]
qmin = 1423
qmax = 20000
list_bom = [41, 1, 40, 4]
list_price = [0.02, 0.23, 0.01, 0.09]
list_moq = [32000, 300, 30000, 300]
list_rv = [2000, 20, 5000, 50]
list_leftover = [14352, 284, 35476, 292]

result = find_satisfied_quantity(list_p, qmin, qmax, list_bom, list_price, list_moq, list_rv, list_leftover)
print(result)


[44000, 1140, 30000, 5400]


In [ ]:
max(math.ceil((1423 * 40 - 35476) / 5000) * 5000, 30000)

30000

In [ ]:
import math

def find_satisfied_quantity(list_p, qmin, qmax, list_bom, list_price, list_moq, list_rv, list_leftover):
    best_q = qmin
    best_scrap_value = float('inf')
    aa = 0

    for q in range(qmin, qmax + 1):
        scrap_value = 0

        for p in range(len(list_p)):
            if list_leftover[p] / list_bom[p] >= q:
                add = 0
            elif list_leftover[p] / list_bom[p] < q:
                add = max(math.ceil((q * list_bom[p] - list_leftover[p]) / list_rv[p]) * list_rv[p], list_moq[p])


            scrap_value += ((list_leftover[p] + add) - q * list_bom[p]) * list_price[p]

        if scrap_value <= 100 and scrap_value < best_scrap_value:
            best_q = q
            best_scrap_value = scrap_value
            add = aa

    return best_q, best_scrap_value, aa

# Example usage:
list_p = [1, 2, 3]
qmin = 1
qmax = 100000
list_bom = [4, 5, 6]
list_price = [0.1, 0.2, 0.3]
list_moq = [100, 200, 300]
list_rv = [10, 20, 30]
list_leftover = [50, 60, 10000]

result = find_satisfied_quantity(list_p, qmin, qmax, list_bom, list_price, list_moq, list_rv, list_leftover)
print(result)


(1716, 1.8, 0)


In [ ]:
def find_satisfied_quantity(list_p, qmin, qmax, list_bom, list_price, list_moq, list_rv, list_leftover):
    for q in range(qmin, qmax+1):
        scrap_value = 0
        list_add = []
        for p in range(len(list_p)):
            if (list_leftover[p] / list_bom[p]) >= q:
                add = 0
            elif (list_leftover[p] / list_bom[p]) < q:
                add = max(math.ceil((q * list_bom[p] - list_leftover[p]) / list_rv[p]) * list_rv[p], list_moq[p])

            scrap_value += ((list_leftover[p] + add) - q * list_bom[p]) * list_price[p]
            list_add.append(add)
        if scrap_value <= 100:
            return list_add

    return None

In [ ]:
data = {
    'SKU': ['61009158', '61009158', '61009158', '61009158'],
    'Material': ['P11212449', 'P11242048', 'P11250879', 'P11260178'],
    'Min Qty (SKU)': [288, 288, 288, 288],
    'Max Qty (SKU)': [20000, 20000, 20000, 20000],
    'BOM': [41, 1, 40, 4],
    'Price ($)': [0.02, 0.23, 0.01, 0.09],
    'MOQ(PM)': [32000, 300, 30000, 300],
    'RV(PM)': [2000, 20, 5000, 50],
    'Available(PM)': [14352, 284, 35476, 292]
    # 'Option 1 Buy': np.nan,
    # 'Option 1 SKU Qty': np.nan
}

dff = pd.DataFrame(data)
dff